# Mushroom Model — Prediction Testing Notebook

Notebook ini digunakan untuk menguji endpoint prediksi model yang sudah di-deploy via **TensorFlow Serving**.

## 1. Import & Konfigurasi

In [22]:
import json
import base64
import requests
import pandas as pd
import numpy as np
import tensorflow as tf

# ── Konfigurasi endpoint TF Serving ──────────────────────────────────────────
TF_SERVING_HOST  = "http://localhost:8501"
MODEL_NAME       = "mushroom_model"
METADATA_URL     = f"{TF_SERVING_HOST}/v1/models/{MODEL_NAME}"

print(f"Metadata URL: {METADATA_URL}")

Metadata URL: http://localhost:8501/v1/models/mushroom_model


## 2. Verifikasi Server Status

In [10]:
try:
    response = requests.get(METADATA_URL, timeout=5)
    data = response.json()
    print("✅ TF Serving berjalan!")
    print(json.dumps(data, indent=2))
except requests.exceptions.ConnectionError:
    print("❌ TF Serving tidak bisa dihubungi.")
    print("   Jalankan: bash serving/docker_commands.sh")
except Exception as e:
    print(f"⚠️  Error: {e}")

✅ TF Serving berjalan!
{
  "model_version_status": [
    {
      "version": "1",
      "state": "AVAILABLE",
      "status": {
        "error_code": "OK",
        "error_message": ""
      }
    }
  ]
}


## 3. Fungsi Helper

Fungsi untuk membuat `tf.train.Example` dari dictionary fitur mushroom,
kemudian serialize ke bytes dan encode Base64 untuk dikirim via REST API.

In [11]:
def make_tf_example(features: dict) -> str:

    feature = {}
    for key, value in features.items():
        feature[key] = tf.train.Feature(
            bytes_list=tf.train.BytesList(
                value=[value.encode('utf-8')]
            )
        )
    
    example = tf.train.Example(
        features=tf.train.Features(feature=feature)
    )
 
    serialized = example.SerializeToString()
    return base64.b64encode(serialized).decode('utf-8')


def predict(samples: list) -> list:

    instances = [
        {"b64": make_tf_example(sample)} for sample in samples
    ]
    payload = {"instances": instances}
    
    response = requests.post(
        PREDICT_URL,
        data=json.dumps(payload),
        headers={"Content-Type": "application/json"},
        timeout=10,
    )
    response.raise_for_status()
    predictions = response.json().get("predictions", [])
    

    results = []
    for prob in predictions:
        p = prob[0] if isinstance(prob, list) else prob
        label = "🍄 BERACUN (poisonous)" if p >= 0.5 else "✅ AMAN (edible)"
        results.append({
            "probability_poisonous": round(float(p), 4),
            "probability_edible": round(1 - float(p), 4),
            "prediction": label,
            "confidence": f"{max(float(p), 1-float(p))*100:.1f}%"
        })
    return results


print("✅ Fungsi helper siap digunakan")

✅ Fungsi helper siap digunakan


## 4. Contoh Data Mushroom

Setiap fitur menggunakan kode singkat sesuai dataset UCI Mushroom.
Referensi kode: https://www.kaggle.com/datasets/uciml/mushroom-classification

In [12]:
sample_poisonous = {
    "cap-shape"                : "x",
    "cap-surface"              : "s",
    "cap-color"                : "n",
    "bruises"                  : "t",
    "odor"                     : "p",
    "gill-attachment"          : "f",
    "gill-spacing"             : "c",
    "gill-size"                : "n",
    "gill-color"               : "k",
    "stalk-shape"              : "e",
    "stalk-root"               : "e",
    "stalk-surface-above-ring" : "s",
    "stalk-surface-below-ring" : "s",
    "stalk-color-above-ring"   : "w",
    "stalk-color-below-ring"   : "w",
    "veil-type"                : "p",
    "veil-color"               : "w",
    "ring-number"              : "o",
    "ring-type"                : "p",
    "spore-print-color"        : "k",
    "population"               : "s",
    "habitat"                  : "u",
}

sample_edible = {
    "cap-shape"                : "b",
    "cap-surface"              : "s",
    "cap-color"                : "y",
    "bruises"                  : "t",
    "odor"                     : "a",
    "gill-attachment"          : "f",
    "gill-spacing"             : "c",
    "gill-size"                : "b",
    "gill-color"               : "k",
    "stalk-shape"              : "e",
    "stalk-root"               : "c",
    "stalk-surface-above-ring" : "s",
    "stalk-surface-below-ring" : "s",
    "stalk-color-above-ring"   : "w",
    "stalk-color-below-ring"   : "w",
    "veil-type"                : "p",
    "veil-color"               : "w",
    "ring-number"              : "o",
    "ring-type"                : "p",
    "spore-print-color"        : "n",
    "population"               : "n",
    "habitat"                  : "g",
}

sample_unknown = {
    "cap-shape"                : "f",
    "cap-surface"              : "y",
    "cap-color"                : "w",
    "bruises"                  : "f",
    "odor"                     : "n",
    "gill-attachment"          : "f",
    "gill-spacing"             : "w",
    "gill-size"                : "b",
    "gill-color"               : "n",
    "stalk-shape"              : "t",
    "stalk-root"               : "b",
    "stalk-surface-above-ring" : "s",
    "stalk-surface-below-ring" : "s",
    "stalk-color-above-ring"   : "p",
    "stalk-color-below-ring"   : "p",
    "veil-type"                : "p",
    "veil-color"               : "w",
    "ring-number"              : "o",
    "ring-type"                : "l",
    "spore-print-color"        : "w",
    "population"               : "v",
    "habitat"                  : "p",
}

samples = [
    sample_poisonous,
    sample_edible,
    sample_unknown
]

labels_expected = [
    "BERACUN",
    "AMAN",
    "?"
]

print(
    f"✅ {len(samples)} sampel siap "
    f"dikirim ke model"
)

✅ 3 sampel siap dikirim ke model


## 5. Kirim Request Prediksi

In [20]:
try:
    results = predict(samples)
    
    for i, (result, expected) in enumerate(zip(results, labels_expected)):
        print(f"── Sampel {i+1} (diharapkan: {expected}) ────────────────────")
        print(f"   Prediksi   : {result['prediction']}")
        print(f"   P(beracun) : {result['probability_poisonous']}")
        print(f"   P(aman)    : {result['probability_edible']}")
        print(f"   Confidence : {result['confidence']}")
        print()

except requests.exceptions.ConnectionError:
    print("❌ Tidak bisa terhubung ke TF Serving.")
    print("   Jalankan: bash serving/docker_commands.sh")
except requests.exceptions.HTTPError as e:
    print(f"❌ HTTP Error: {e}")
    print(f"   Response: {e.response.text}")

── Sampel 1 (diharapkan: BERACUN) ────────────────────
   Prediksi   : 🍄 BERACUN (poisonous)
   P(beracun) : 0.9998
   P(aman)    : 0.0002
   Confidence : 100.0%

── Sampel 2 (diharapkan: AMAN) ────────────────────
   Prediksi   : ✅ AMAN (edible)
   P(beracun) : 0.0
   P(aman)    : 1.0
   Confidence : 100.0%

── Sampel 3 (diharapkan: ?) ────────────────────
   Prediksi   : ✅ AMAN (edible)
   P(beracun) : 0.0026
   P(aman)    : 0.9974
   Confidence : 99.7%



## 6. Visualisasi Hasil dalam Tabel

In [14]:
try:
    results = predict(samples)
    
    df = pd.DataFrame([
        {
            "Sampel": f"Sampel {i+1}",
            "Label Asli": labels_expected[i],
            "Prediksi": "BERACUN" if results[i]["probability_poisonous"] >= 0.5 else "AMAN",
            "P(Beracun)": results[i]["probability_poisonous"],
            "P(Aman)": results[i]["probability_edible"],
            "Confidence": results[i]["confidence"],
            "Status": "✅" if (
                (results[i]["probability_poisonous"] >= 0.5 and labels_expected[i] == "BERACUN") or
                (results[i]["probability_poisonous"] < 0.5  and labels_expected[i] == "AMAN")
            ) else "❓",
        }
        for i, _ in enumerate(samples)
    ])
    
    display(df)
    
except Exception as e:
    print(f"Error: {e}")

,Sampel,Label Asli,Prediksi,P(Beracun),P(Aman),Confidence,Status
0,Sampel 1,BERACUN,BERACUN,0.9998,0.0002,100.0%,✅
1,Sampel 2,AMAN,AMAN,0.0000,1.0000,100.0%,✅
2,Sampel 3,?,AMAN,0.0026,0.9974,99.7%,❓


## 7. Contoh JSON Request Manual (via cURL)

In [18]:
import json, base64, requests, tensorflow as tf

PREDICT_URL = "http://localhost:8501/v1/models/mushroom_model:predict"

def make_tf_example(features):
    feature = {
        k: tf.train.Feature(bytes_list=tf.train.BytesList(value=[v.encode()]))
        for k, v in features.items()
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return base64.b64encode(example.SerializeToString()).decode()

sample = {
    "cap-shape": "x", "cap-surface": "s", "cap-color": "n",
    "bruises": "t", "odor": "p", "gill-attachment": "f",
    "gill-spacing": "c", "gill-size": "n", "gill-color": "k",
    "stalk-shape": "e", "stalk-root": "e",
    "stalk-surface-above-ring": "s", "stalk-surface-below-ring": "s",
    "stalk-color-above-ring": "w", "stalk-color-below-ring": "w",
    "veil-type": "p", "veil-color": "w", "ring-number": "o",
    "ring-type": "p", "spore-print-color": "k",
    "population": "s", "habitat": "u"
}

payload = json.dumps({"instances": [{"b64": make_tf_example(sample)}]})
response = requests.post(PREDICT_URL, data=payload, headers={"Content-Type": "application/json"})

prob = response.json()["predictions"][0][0]
label = "🍄 BERACUN" if prob >= 0.5 else "✅ AMAN"
print(f"Probabilitas beracun : {prob:.4f}")
print(f"Hasil prediksi       : {label}")

Probabilitas beracun : 0.9998
Hasil prediksi       : 🍄 BERACUN


## 8. Batch Testing — Multiple Sampel Sekaligus

In [16]:
BATCH_SIZE = 5

batch_samples = [
    sample_poisonous, sample_edible, sample_unknown,
    sample_poisonous, sample_edible
][:BATCH_SIZE]

try:
    print(f"📡 Mengirim batch {len(batch_samples)} sampel sekaligus...")
    batch_results = predict(batch_samples)
    
    print(f"✅ Berhasil mendapat {len(batch_results)} prediksi\n")
    for i, r in enumerate(batch_results):
        print(f"  [{i+1}] {r['prediction']} — confidence: {r['confidence']}")

except Exception as e:
    print(f"Error: {e}")

📡 Mengirim batch 5 sampel sekaligus...
✅ Berhasil mendapat 5 prediksi

  [1] 🍄 BERACUN (poisonous) — confidence: 100.0%
  [2] ✅ AMAN (edible) — confidence: 100.0%
  [3] ✅ AMAN (edible) — confidence: 99.7%
  [4] 🍄 BERACUN (poisonous) — confidence: 100.0%
  [5] ✅ AMAN (edible) — confidence: 100.0%
